In [1]:
# Run this single cell in Google Colab or Kaggle (with T4 GPU enabled)
!pip install -q git+https://github.com/huggingface/transformers.git accelerate fastapi uvicorn pillow numpy
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import base64, io, subprocess, threading, time, re
import numpy as np
import torch
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from transformers import Sam3Processor, Sam3Model

MODEL_ID = "unsloth/sam3.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

print(f"Loading {MODEL_ID} on {device} ({dtype})...")
processor = Sam3Processor.from_pretrained(MODEL_ID)
model = Sam3Model.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()
print("SAM 3.1 loaded Ready!")

app = FastAPI()

class SegmentRequest(BaseModel):
    image_b64: str
    prompts: list[str]
    threshold: float = 0.45

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": device}

@app.post("/segment")
def segment(req: SegmentRequest):
    img_bytes = base64.b64decode(req.image_b64)
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    detections = []

    for prompt in req.prompts:
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, dtype)
        with torch.inference_mode():
            outputs = model(**inputs)
        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=req.threshold,
            mask_threshold=0.5,
            target_sizes=inputs.get("original_sizes").tolist(),
        )[0]

        if len(results["masks"]) > 0:
            best_idx = int(torch.argmax(results["scores"]).item())
            mask_np = results["masks"][best_idx].cpu().numpy().astype(np.uint8) * 255
            box = [int(x) for x in results["boxes"][best_idx].cpu().tolist()]
            score = float(results["scores"][best_idx].cpu().item())

            buf = io.BytesIO()
            Image.fromarray(mask_np).save(buf, format="PNG")
            mask_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

            detections.append({
                "label": prompt,
                "score": score,
                "bbox_2d": box,
                "mask_png_b64": mask_b64,
            })

    return {"model": MODEL_ID, "detections": detections}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8005, log_level="warning"), daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8005"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n========================================================")
        print(f"✅ PASTE THIS INTO YOUR LOCAL .env AS SAM31_ENDPOINT_URL:")
        print(f"SAM31_ENDPOINT_URL={match.group(0)}")
        print("========================================================\n")
        break

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Loading unsloth/sam3.1 on cpu (torch.float16)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


processor_config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.68GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

SAM 3.1 loaded Ready!

✅ PASTE THIS INTO YOUR LOCAL .env AS SAM31_ENDPOINT_URL:
SAM31_ENDPOINT_URL=https://slots-strengths-timber-progressive.trycloudflare.com

